In [ ]:
import torch
from inference import get_model
import cv2
import os
from paddleocr import PaddleOCR
from time import perf_counter as t
from transformers import (
    AutoImageProcessor,
    AutoModelForTextRecognition,
)
import dotenv 
dotenv.load_dotenv()
from PIL import Image
os.environ["FLAGS_enable_pir_api"] = "0"

# Roboflow Universe model
model = get_model(
    model_id="indian-license-plate-detection-6tmbr-b9bnb-nfk37/1",
    api_key=os.environ["ROBO_KEY"]
)

image = cv2.imread("frame_3s.png")

results = model.infer(image)

print(results)
print(model)

image = cv2.imread("frame_3s.png")

s = t()
results = model.infer(image)
e = t()

print("Inference time:", e - s)

result = results[0]

for i, p in enumerate(result.predictions):

    print(
        p.class_name,
        p.confidence,
        p.x,
        p.y,
        p.width,
        p.height
    )

    # Center → corner coordinates
    x1 = int(p.x - p.width / 2)
    y1 = int(p.y - p.height / 2)
    x2 = int(p.x + p.width / 2)
    y2 = int(p.y + p.height / 2)

    # Clamp to image boundaries
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(image.shape[1], x2)
    y2 = min(image.shape[0], y2)

    # Extract only detected region
    plate_crop = image[y1:y2, x1:x2]

    if plate_crop.size == 0:
        continue

    # Save only the extracted plate
    cv2.imwrite(f"plate_{i}.jpg", plate_crop)

    print(f"Saved: plate_{i}.jpg")
    print("Crop size:", plate_crop.shape)


# -----------------------------
# Load PaddleOCR
# -----------------------------
ocr = PaddleOCR(
    lang="en",
    device="cpu",
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
# -----------------------------
# Load extracted plate image
# -----------------------------
plate = cv2.imread("result.jpg")

if plate is None:
    raise FileNotFoundError("plate_crop.jpg not found")

# Upscale small plate crops
plate = cv2.resize(
    plate,
    None,
    fx=4,
    fy=4,
    interpolation=cv2.INTER_CUBIC,
)

# -----------------------------
# OCR
# -----------------------------
results = ocr.predict(plate)

# Print everything
for result in results:
    result.print()

    # Extract recognized text
    data = result.json
    if callable(data):
        data = data()

    texts = data.get("rec_texts", [])
    scores = data.get("rec_scores", [])

    for text, score in zip(texts, scores):
        print(f"TEXT: {text}")
        print(f"CONFIDENCE: {score:.3f}")


device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL = "PaddlePaddle/PP-OCRv6_tiny_rec_safetensors"

processor = AutoImageProcessor.from_pretrained(MODEL)
model = AutoModelForTextRecognition.from_pretrained(MODEL).to(device)
model.eval()

image = Image.open("plate_0.jpg").convert("RGB")

inputs = processor(
    images=image,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    s = t()
    outputs = model(**inputs)
    e = t()
print(e-s)
results = processor.post_process_text_recognition(outputs)

for result in results:
    print(result)